In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Warning message in install.packages("corrplot"):
“installation of package ‘corrplot’ had non-zero exit status”


In [ ]:
# Installation des librairies nécessaires (si besoin)
install.packages(c("ggplot2", "corrplot", "caret", "pROC", "dplyr", "tidyr"), repos='http://cran.us.r-project.org', dependencies=TRUE)

library(ggplot2)
library(corrplot)
library(caret)
library(pROC)
library(dplyr)
library(tidyr)

# Chargement des données
# Remplace le chemin par le tien
dataset <- read.csv("/content/drive/MyDrive/Colab Notebook/Data_valo/original_dataset.csv")

# Configuration des couleurs pour les graphiques (cohérence visuelle pour la vidéo)
cols_fill <- c("#E74C3C", "#3498DB") # Rouge (Non remboursé) / Bleu (Remboursé)
theme_set(theme_minimal()) # Thème propre

ERROR: Error in library(corrplot): there is no package called ‘corrplot’


In [ ]:
# @title 3. Description des données

cat("=== Dimensions du Dataset ===\n")
cat("Lignes (Observations) :", nrow(dataset), "\n")
cat("Colonnes (Variables) :", ncol(dataset), "\n\n")

cat("=== Types des variables ===\n")
str(dataset)

cat("\n=== Aperçu des données ===\n")
head(dataset)

In [ ]:
# @title 4. Qualité des données

# --- A. Gestion des valeurs manquantes ---
missing_vals <- colSums(is.na(dataset))
cat("Valeurs manquantes par colonne :\n")
print(missing_vals[missing_vals > 0])

# Nettoyage basique (suppression des lignes avec NA pour l'exemple)
dataset <- na.omit(dataset)

# --- B. Détection et Suppression des Outliers (Mahalanobis) ---
# Sélection des variables numériques
nums <- dataset %>% select_if(is.numeric)

# Calcul distance de Mahalanobis
md <- mahalanobis(nums, colMeans(nums), cov(nums))
cutoff <- qchisq(0.975, df = ncol(nums)) # Seuil de 2.5%
outliers <- which(md > cutoff)

cat("\nNombre d'outliers détectés et supprimés :", length(outliers), "\n")

# Suppression des outliers
dataset_clean <- dataset[-outliers, ]

# --- C. Visualisation Univariée (Distribution) ---
# Graphique pour montrer la répartition de la cible (Important pour voir le déséquilibre)
dataset_clean$paid_label <- factor(dataset_clean$loan_paid_back,
                                   levels=c(0,1),
                                   labels=c("Défaut (0)", "Remboursé (1)"))

ggplot(dataset_clean, aes(x=paid_label, fill=paid_label)) +
  geom_bar(width=0.6, color="black", alpha=0.8) +
  scale_fill_manual(values=cols_fill) +
  labs(title="Répartition de la variable cible (Remboursement)",
       y="Nombre de prêts", x="") +
  geom_text(stat='count', aes(label=..count..), vjust=-0.5)

In [ ]:
# @title 5. Analyse Statistique Multivariée

# Calcul de la corrélation
nums_clean <- dataset_clean %>% select_if(is.numeric)
cor_matrix <- cor(nums_clean)

# Visualisation propre
corrplot(cor_matrix,
         method = "color",
         type = "upper",
         tl.col = "black",
         tl.cex = 0.7,
         addCoef.col = "black", # Ajoute les chiffres
         number.cex = 0.5,
         diag = FALSE,
         title = "Matrice de Corrélation")

In [ ]:
# Création du graphique empilé 100%
ggplot(dataset_clean, aes(x = grade_subgrade, fill = paid_label)) +
  geom_bar(position = "fill", color = "black", alpha = 0.8) +
  scale_fill_manual(values = cols_fill) +
  labs(title = "Probabilité de défaut par Note (Grade)",
       subtitle = "Les grades A/B remboursent mieux que les grades E/F",
       x = "Note du prêt (Grade)",
       y = "Proportion",
       fill = "Statut") +
  scale_y_continuous(labels = scales::percent) +
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1))

In [ ]:
ggplot(dataset_clean, aes(x = loan_amount, y = interest_rate, color = paid_label)) +
  geom_point(alpha = 0.4, size = 1.5) +
  scale_color_manual(values = cols_fill) +
  labs(title = "Relation : Montant vs Taux d'intérêt",
       subtitle = "Les taux élevés (axe Y haut) concentrent plus de points rouges",
       x = "Montant du prêt ($)",
       y = "Taux d'intérêt (%)",
       color = "Statut") +
  theme(legend.position = "top")

In [ ]:
# Préparation simple pour le modèle
# On garde les variables numériques pour simplifier l'exemple vidéo
X <- nums_clean %>% select(-loan_paid_back)
y <- as.factor(nums_clean$loan_paid_back)
levels(y) <- c("No", "Yes") # Requis pour caret

# Split Train/Test
set.seed(123)
index <- createDataPartition(y, p=0.8, list=FALSE)
train_data <- data.frame(X[index, ], Class = y[index])
test_x <- X[-index, ]
test_y <- y[-index]

# Entraînement Rapide (Régression Logistique)
model <- train(Class ~ ., data=train_data, method="glm", family="binomial",
               trControl = trainControl(method = "none"))

# Prédictions
probs <- predict(model, test_x, type = "prob")

# Courbe ROC
roc_obj <- roc(test_y, probs$Yes, levels=c("No", "Yes"))

# Affichage Graphique ROC
ggroc(roc_obj, legacy.axes = TRUE) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "gray") +
  labs(title = paste("Courbe ROC - AUC =", round(auc(roc_obj), 3)),
       subtitle = "Capacité du modèle à distinguer les bons des mauvais payeurs",
       x = "1 - Spécificité (Taux de Faux Positifs)",
       y = "Sensibilité (Taux de Vrais Positifs)") +
  style_roc()